In [13]:
from axetract.preprocessor.axe_preprocessor import AXEPreprocessor

In [13]:
dummy_html = """
<html>
    <head><title>Product Page</title></head>
    <body>
        <nav><ul><li>Home</li><li>Products</li></ul></nav>
        <div id="main-content" class="container" style="background: #fff;">
            <h1>SuperWidget 3000</h1>
            <p class="description">The SuperWidget 3000 is the best tool for <b>AI engineers</b>. 
            It features high-performance processing and low latency.</p>
            <table class="specs-table">
                <tr><th>Feature</th><th>Value</th></tr>
                <tr><td>Weight</td><td>1.2kg</td></tr>
                <tr><td>Price</td><td>$299</td></tr>
            </table>
            <div class="reviews">
                <h2>User Reviews</h2>
                <div class="review">"Life changing tool!" - Jane Doe</div>
                <div class="review">"Fast but expensive." - John Smith</div>
            </div>
        </div>
        <footer>Copyright 2025 WidgetCorp</footer>
    </body>
</html>
"""

# The query to use later with the LLM
query = "Extract the price and weight of the SuperWidget 3000 from the following context."

In [15]:
from axetract.data_types import AXESample

test = AXESample(id="1", query=query, is_content_url=False, content=dummy_html)

In [16]:
pre = AXEPreprocessor()
res = pre(test)
res

[AXESample(id='1', content='\n<html>\n    <head><title>Product Page</title></head>\n    <body>\n        <nav><ul><li>Home</li><li>Products</li></ul></nav>\n        <div id="main-content" class="container" style="background: #fff;">\n            <h1>SuperWidget 3000</h1>\n            <p class="description">The SuperWidget 3000 is the best tool for <b>AI engineers</b>. \n            It features high-performance processing and low latency.</p>\n            <table class="specs-table">\n                <tr><th>Feature</th><th>Value</th></tr>\n                <tr><td>Weight</td><td>1.2kg</td></tr>\n                <tr><td>Price</td><td>$299</td></tr>\n            </table>\n            <div class="reviews">\n                <h2>User Reviews</h2>\n                <div class="review">"Life changing tool!" - Jane Doe</div>\n                <div class="review">"Fast but expensive." - John Smith</div>\n            </div>\n        </div>\n        <footer>Copyright 2025 WidgetCorp</footer>\n    </bo

In [17]:
res[0].chunks

[AXEChunk(chunkid='0-1', content='<html>\n<head><title>Product Page</title>\n</head><body>\n<ul><li>Home</li><li>Products</li></ul>\n<div>\n<h1>SuperWidget 3000</h1>\n<p>The SuperWidget 3000 is the best tool for <b>AI engineers</b>. \n            It features high-performance processing and low latency.</p>\n<table>\n<tr><th>Feature</th><th>Value</th></tr>\n<tr><td>Weight</td><td>1.2kg</td></tr>\n<tr><td>Price</td><td>$299</td></tr>\n</table>\n<div>\n<h2>User Reviews</h2>\n<div>"Life changing tool!" - Jane Doe</div>\n<div>"Fast but expensive." - John Smith</div>\n</div>\n</div>\n<footer>Copyright 2025 WidgetCorp</footer>\n</body>\n</html>')]

In [18]:
from axetract.pruner.axe_pruner import AXEPruner

In [19]:
# from axetract.llm.vllm_client import LocalVLLMClient
from axetract.llm.hf_client import HuggingFaceClient

In [20]:
# lc = LocalVLLMClient(
#     config={
#         "model_name": "Qwen/Qwen3-0.6B",
#         # api_key:"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",
#         "max_tokens": 1024,  # max new tokens
#         "engine_args": {
#             "gpu_memory_utilization": 0.7,
#             "max_model_len": 6000,  # max total tokens
#             # "enforce_eager": True,
#         },
#         "temperature": 0.0,
#         "lora_modules": {
#             "pruner": {
#                 "path": "abdo-Mansour/Pruner_Adaptor_Qwen_3_FINAL_EXTRA",
#                 "temperature": 0.0,
#             },
#             "qa": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_QA_websrc", "temperature": 1.0},
#             "schema": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_Final", "temperature": 0.0},
#         },
#     }
# )
lc =HuggingFaceClient(
    config={
        "model_name": "Qwen/Qwen3-0.6B",
        # api_key:"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",
        "max_tokens": 1024,  # max new tokens
        "engine_args": {
            "gpu_memory_utilization": 0.7,
            "max_model_len": 6000,  # max total tokens
            # "enforce_eager": True,
        },
        "temperature": 0.0,
        "lora_modules": {
            "pruner": {
                "path": "abdo-Mansour/Pruner_Adaptor_Qwen_3_FINAL_EXTRA",
                "temperature": 0.0,
            },
            "qa": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_QA_websrc", "temperature": 1.0},
            "schema": {"path": "abdo-Mansour/Extractor_Adaptor_Qwen3_Final", "temperature": 0.0},
        },
    }

)

`torch_dtype` is deprecated! Use `dtype` instead!
[2026-03-13 18:50:50] INFO modeling.py:987: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


adapter_config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

In [21]:
from axetract.prompts.pruner_prompt import PRUNER_PROMPT
from axetract.prompts.qa_prompt import QA_PROMPT
from axetract.prompts.schema_prompt import SCHEMA_PROMPT

In [22]:
pru = AXEPruner(
    # llm_client=lc,
    llm_pruner_client=lc,
    llm_pruner_prompt=PRUNER_PROMPT,
)

In [23]:
from axetract.extractor.axe_extractor import AXEExtractor
ext = AXEExtractor(
    llm_extractor_client=lc,
    schema_generation_prompt_template=SCHEMA_PROMPT,
    query_generation_prompt_template=QA_PROMPT,
)

In [ ]:
from axetract.pipeline import AXEPipeline
from axetract.postprocessor.axe_postprocessor import AXEPostprocessor
pipeline = AXEPipeline(
    preprocessor=AXEPreprocessor(),
    pruner=pru,
    extractor=ext,
    postprocessor=AXEPostprocessor()
)




The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[2026-03-13 18:51:44] WARNING html_util.py:362: FALLBACK USED


AXEResult(id='08efd0dc-07b3-480f-8b2d-f58889707766', prediction='150.00', xpaths=None, status=<Status.SUCCESS: 'success'>, error=None)

In [25]:
result = pipeline.process(
    input_data=test.original_html,
    query=query
)

[2026-03-13 19:33:56] WARNING html_util.py:362: FALLBACK USED


In [ ]:
print(result.xpaths) 

None


In [17]:
from axetract import AXEPipeline

pipeline = AXEPipeline.from_config()

result = pipeline.process(
    input_data="https://en.wikipedia.org/wiki/Napoleonic_Wars",
    query="when did the wars start?"
)

In [18]:
# for article in result.prediction['articles']:
#     print(f"{article['title']} ({article['points']} pts)")
print(result)

id='5394cf65-e375-426d-ad9d-d230321b99ac' prediction='18 May 1803 – 20 November 1815' xpaths=None status=<Status.SUCCESS: 'success'> error=None
